## uwtools + ecFlow

The notebook demonstrates the use of `uwtools` to configure and run an ecFlow server, to generate an ecFlow workflow (a suite definition an `.ecf` files), and to load the suite into the server and execute it. As a demo application, we perform verification of 2-meter temperature at the 6-hour leadtime of a GFS forecast against a GFS leadtime-zero forecast as a source of truth.

First, get to the correct working directory and activate the virtual environment for the demo:

In [1]:
cd $BASEDIR
source conda/etc/profile.d/conda.sh
conda activate demo

Here's the `uwtools`-enabled configuration file for the demo. It's not necessary to understand all of it now. The top-level `app` block defines various application-level items that are useful throughout the workflow. The `ecflow` block defines server and suite-definition blocks specific to ecFlow. And the `wxvx` block defines parameters for verification to be performed by [wxvx](https://github.com/NOAA-GSL/wxvx).

One thing to note looking at the full config is the use of [Jinja2](https://jinja.palletsprojects.com/en/stable/templates/) expressions to correctly tie together various config blocks.

In [2]:
cat config.yaml

app:
  basedir: '{{ "BASEDIR" | env }}'
  cycle: 2026-07-04T12:00:00
  fcst:
    fn: gfs.t{{ app.hh }}z.pgrb2.1p00.f{{ "%03d" % app.leadtime }}
    url: '{{ app.prefix }}/{{ app.fcst.fn }}'
  hh: '{{ app.cycle.strftime("%H") }}'
  leadtime: 6
  prefix: https://nomads.ncep.noaa.gov/pub/data/nccf/com/gfs/prod/gfs.{{ app.yyyymmdd }}/{{ app.hh }}/atmos
  yyyymmdd: '{{ app.cycle.strftime("%Y%m%d") }}'
ecflow:
  server:
    ECF_HOME: '{{ app.basedir }}'
  suitedef:
    suite_vx:
      task_extract_grids:
        script:
          body: |
            init $$
            uw config realize -i {{ app.basedir }}/config.yaml --key-path wxvx >{{ app.basedir }}/vx.yaml
            wxvx -c {{ app.basedir }}/vx.yaml -t grids
          includes:
            entry: [common.h, server.h]
          manual: Extract forecast and truth grids
        trigger: /vx/fetch_truth == complete
      task_fetch_truth:
        script:
          body: |
            init $$
            test -e {{ app.fcst.fn }} || wget -

The `ecflow.server` block provides the bare mininum for running the ecFlow server: A definition of the `ECF_HOME` environment variable, which specifies where the server should run and where, by default, it will look for `.h` include files when creating job scripts from `.ecf` files:

In [3]:
uw config realize -i config.yaml --key-path ecflow.server

ECF_HOME: /home/maddenp/git/uwtools/wip


In this case, `ecflow.server` refers to `app.basedir` for its value, and `app.basedir` refers to the environment variable `BASEDIR` set by the `start` script.

Start the ecFlow server using the config specified in `config.yaml`. Request a JSON-formatted report of important server environment variables, redirect that to `server.json`, and redirect the rest of the output to `server.log`:

In [4]:
uw ecflow server -c config.yaml --report >server.json 2>server.log &

[1] 28827


Wait until the server has started up and written `server.json`:

In [5]:
while [[ ! -s server.json ]]; do sleep 1; done

View `server.log` and note that there were no errors, and the the server is using a random available TCP port.

Also note that, since `~/.ecflowrc/ssl` did not already contain the SSL certificate files required for secure communication with the ecFlow server, these were generated.

In [6]:
cat server.log

[2026-07-08T03:43:27]     INFO Validating config against internal schema: ecflow
[2026-07-08T03:43:27]     INFO Schema validation succeeded for ecFlow config
[2026-07-08T03:43:27]     INFO Generating SSL private key: /home/maddenp/.ecflowrc/ssl/server.key
[2026-07-08T03:43:28]     INFO Generating SSL certificate: /home/maddenp/.ecflowrc/ssl/server.crt
[2026-07-08T03:43:28]     INFO Generating DH parameters: /home/maddenp/.ecflowrc/ssl/dh2048.pem
[2026-07-08T03:43:30]     INFO SSL certificate files written to /home/maddenp/.ecflowrc/ssl
[2026-07-08T03:43:30]     INFO Server started on port 14332


At server-startup time, `uw ecflow server` also creates a `.h` include file called `server.h`, defining environment variables that will be needed by `.ecf` files:

In [7]:
cat server.h

export ECF_HOST=%ECF_HOST%
export ECF_NAME=%ECF_NAME%
export ECF_PASS=%ECF_PASS%
export ECF_PORT=%ECF_PORT%
export ECF_TRYNO=%ECF_TRYNO%
export PATH=/home/maddenp/git/uwtools/wip/conda/envs/demo/bin/:$PATH


The `server.json` file, containing current values for some useful server-related environment variables, was also created:

In [8]:
jq . server.json

{
  "ECF_HOME": "/home/maddenp/git/uwtools/wip",
  "ECF_HOST": "xix",
  "ECF_PORT": "14332",
  "ECF_SSL": "1"
}


Use `jq` to extract values from this file to use to set some crucial environment variables that will allow the ecFlow client tool, `ecflow_client`, to successfully communicate with the server:

In [9]:
export ECF_PORT=$(jq -r .ECF_PORT server.json)
export ECF_SSL=$(jq -r .ECF_SSL server.json)

With these environment variables in place, use `ecflow_client` to ping the server and verify that it is listening:

In [10]:
ecflow_client --ping

ping server(localhost:14332) succeeded in 00:00:00.007415  ~7 milliseconds


By default, the server starts halted, so place it into running state with the `restart` command:

In [11]:
ecflow_client --stats | grep Status
ecflow_client --restart
ecflow_client --stats | grep Status

  Status                          HALTED
  Status                          RUNNING


Now the server is ready to be loaded with a suite definition, provided by the `ecflow.suitedef` block in `config.yaml`. The `uw ecflow realize` command creates a `suite.def` file in the directory specified by `--output-dir` and, under the directory specified by `--scripts-dir`, any `.ecf` files defined by `script` blocks in the suite definition:

In [12]:
uw ecflow realize -c config.yaml --output-dir=$BASEDIR --scripts-dir=$BASEDIR

[2026-07-08T03:43:43]     INFO Validating config against internal schema: ecflow
[2026-07-08T03:43:43]     INFO Schema validation succeeded for ecFlow config


Here is the `suite.def`, in ecFlow's custom configuration language.

Note that the `fetch_truth` task can proceed immediately, but that `extract_grids` truth requires `fetch_truth` to be complete, as it uses the fetched truth data as an input; and that `verification` requires `extract_grids` to be complete, as it uses the extracted grids in verification:

In [13]:
cat suite.def

#5.16.0
suite vx
  edit ECF_JOB_CMD '/usr/bin/env -S bash %ECF_JOB% >%ECF_JOBOUT% 2>&1'
  task extract_grids
    trigger /vx/fetch_truth == complete
  task fetch_truth
endsuite
# enddef


The `.ecf` files, which are templates used to create executable job scripts that run ecFlow tasks, are created in a directory hierarchy based on the suite name, family names (this demo does not use families), and task names:

In [14]:
tree vx

vx
├── extract_grids.ecf
└── fetch_truth.ecf

1 directory, 2 files


The `fetch_truth.ecf` file defines a task that will download truth data to verify a forecast against. It defines some help text between the `%manual` and `%end` tags that can be viewed, for example, in the ecFlow GUI application, `ecflow_ui`. When rendered by the server to a job script, the contents of the `server.h` and `common.h` include files will be inlined where the `%include` statements appear.

In [15]:
cat vx/fetch_truth.ecf

%manual
Fetch GRIB truth data
%end

%include common.h

%include server.h

init $$
test -e gfs.t12z.pgrb2.1p00.f006 || wget --quiet https://nomads.ncep.noaa.gov/pub/data/nccf/com/gfs/prod/gfs.20260704/12/atmos/gfs.t12z.pgrb2.1p00.f006


We saw the contents of the `server.h` file generated by `uw ecflow server` earlier. Here's `common.h`, which belongs to this demo application and defines some functions and traps that allow task jobs to communicate their status to the server:

In [16]:
cat common.h

abort() {
  trap EXIT # unset previous trap
  ecflow_client --abort=$ECF_RID
  exit 1
}

trap abort ERR SIGTERM

complete() {
  ecflow_client --ssl --complete
}

trap complete EXIT

init() {
  export ECF_RID=$1
  ecflow_client --ssl --init=$ECF_RID
}

set -euxo pipefail


Here's the `.ecf` file that runs `wxvx` to extract grids from the forecast and truth datasets to be used in verification:

In [17]:
cat vx/extract_grids.ecf

%manual
Extract forecast and truth grids
%end

%include common.h

%include server.h

init $$
uw config realize -i /home/maddenp/git/uwtools/wip/config.yaml --key-path wxvx >/home/maddenp/git/uwtools/wip/vx.yaml
wxvx -c /home/maddenp/git/uwtools/wip/vx.yaml -t grids


Load the suite into the server:

In [18]:
ecflow_client --load=suite.def

Verify that the suite was loaded:

In [19]:
ecflow_client --get

#5.16.0
suite vx
  edit ECF_JOB_CMD '/usr/bin/env -S bash %ECF_JOB% >%ECF_JOBOUT% 2>&1'
  task extract_grids
    trigger /vx/fetch_truth == complete
  task fetch_truth
endsuite
# enddef


Begin suite execution:

In [20]:
ecflow_client --begin=vx

Wait for the tasks to complete:

In [21]:
while true; do
  state=$(ecflow_client --query state /vx/fetch_truth)
  echo "  fetch_truth: $state"
  state=$(ecflow_client --query state /vx/extract_grids)
  echo "extract_grids: $state"
  test $state == complete && break
  sleep 1
done

  fetch_truth: complete
extract_grids: submitted
  fetch_truth: complete
extract_grids: active
  fetch_truth: complete
extract_grids: active
  fetch_truth: complete
extract_grids: active
  fetch_truth: complete
extract_grids: complete


Shut down the server:

In [22]:
ecflow_client --group="halt=yes; check_pt; terminate=yes"